# 05 Rule-Based Scoring

**Input:** `data/processed/03_user_stories_features.csv` from notebook 03.

**Goal:** apply the scoring methodology defined in notebook 04 to all 31,394 stories. Produce a scored CSV with 5 dimension scores, an overall quality score, a readable score band, and a list of issue tags per story.

**References:**
- The scoring functions live in `src/scoring.py`, written by notebook 04.
- Methodology and weights are grounded in Cohn (INVEST, classic template), Lucassen et al. (2016) (QUS framework), Mordal et al. (2012) (Squale aggregation), and Challa et al. (2011) (interpretation bands).

**Output:** `data/processed/05_user_stories_scored.csv`.

## Plan

1. Load the feature CSV from notebook 03.
2. Import the scoring module.
3. Apply each scoring function across all stories. Add 8 new columns.
4. Look at the distribution of each dimension and the overall score.
5. Count the most common issue tags. This tells us where the bulk of the problems are.
6. Compute per-project average scores. This identifies the best and worst teams in the dataset.
7. Save the scored CSV.


In [2]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
sys.path.insert(0, str(Path('../src').resolve()))
import scoring
print(f"Scoring module loaded from: {scoring.__file__}")
print(f"Functions available: {[f for f in dir(scoring) if not f.startswith('_')]}")

# Load the feature CSV from notebook 03
IN_PATH  = Path('../data/processed/03_user_stories_features.csv')
OUT_PATH = Path('../data/processed/05_user_stories_scored.csv')
df = pd.read_csv(IN_PATH, low_memory=False)
for col in ['Creation_Date', 'Resolution_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
print(f"\nLoaded: {IN_PATH}")
print(f"Shape:  {df.shape}")

Scoring module loaded from: C:\Users\ferdi\Desktop\Requirement Quality Analytics Dashboard\AI-Supported-Requirement-Quality-Analytics-Dashboard\src\scoring.py
Functions available: ['Any', 'Dict', 'List', 'business_value_score', 'clamp', 'clarity_score', 'completeness_score', 'issue_tags', 'math', 'overall_quality_score', 'scope_risk_score', 'score_band', 'testability_score']

Loaded: ..\data\processed\03_user_stories_features.csv
Shape:  (31394, 46)


In [3]:
# Apply all scoring functions to every story.

import time
print("Applying scoring functions to 31,394 stories...")
t0 = time.time()
df['clarity_score']        = df.apply(scoring.clarity_score,        axis=1)
df['completeness_score']   = df.apply(scoring.completeness_score,   axis=1)
df['testability_score']    = df.apply(scoring.testability_score,    axis=1)
df['business_value_score'] = df.apply(scoring.business_value_score, axis=1)
df['scope_risk_score']     = df.apply(scoring.scope_risk_score,     axis=1)
df['overall_quality_score']= df.apply(scoring.overall_quality_score,axis=1)
df['score_band']           = df['overall_quality_score'].apply(scoring.score_band)
df['issue_tags']           = df.apply(scoring.issue_tags,           axis=1)
df['issue_tags_str'] = df['issue_tags'].apply(lambda lst: '|'.join(sorted(lst)) if lst else '')
df['issue_tags_count'] = df['issue_tags'].apply(len)
elapsed = time.time() - t0

print(f"Done in {elapsed:.1f} seconds")
print(f"New columns added: {df.shape[1] - 46}")
print(f"DataFrame shape now: {df.shape}")

Applying scoring functions to 31,394 stories...
Done in 7.1 seconds
New columns added: 10
DataFrame shape now: (31394, 56)


In [4]:
print("OVERALL QUALITY SCORE DISTRIBUTION \n")
print(df['overall_quality_score'].describe().round(2))
print("\n=SORE BAND DISTRIBUTION \n")
band_counts = df['score_band'].value_counts()
band_pct = df['score_band'].value_counts(normalize=True) * 100
band_table = pd.DataFrame({'count': band_counts, 'share_pct': band_pct.round(1)})
band_order = ['Very good', 'Good', 'Average', 'Poor', 'Very poor']
band_table = band_table.reindex(band_order)
print(band_table)

OVERALL QUALITY SCORE DISTRIBUTION 

count    31394.00
mean         1.93
std          0.76
min          0.07
25%          1.85
50%          2.03
75%          2.29
max          5.00
Name: overall_quality_score, dtype: float64

=SORE BAND DISTRIBUTION 

            count  share_pct
score_band                  
Very good     246        0.8
Good         1221        3.9
Average     14360       45.7
Poor        11570       36.9
Very poor    3997       12.7


In [5]:
print("PER DIMENSION SCORE DISTRIBUTION \n")
dim_cols = ['clarity_score', 'completeness_score', 'testability_score',
            'business_value_score', 'scope_risk_score', 'overall_quality_score']
dim_stats = df[dim_cols].describe().round(2).T
dim_stats = dim_stats[['mean', '50%', 'std', 'min', 'max']]
dim_stats.columns = ['mean', 'median', 'std', 'min', 'max']
print(dim_stats)
print("\n DIMENSION RANKING BY MEAN (lowest first = weakest)")
dim_means = df[dim_cols[:-1]].mean().sort_values()
for dim, mean in dim_means.items():
    print(f"  {dim:<28} mean={mean:.2f}")

PER DIMENSION SCORE DISTRIBUTION 

                       mean  median   std   min  max
clarity_score          2.55    2.50  0.55  0.25  5.0
completeness_score     1.77    2.00  0.80  0.00  5.0
testability_score      2.23    2.50  0.62  0.00  5.0
business_value_score   0.94    1.00  0.77  0.00  5.0
scope_risk_score       4.26    4.50  0.81  0.00  5.0
overall_quality_score  1.93    2.03  0.76  0.07  5.0

 DIMENSION RANKING BY MEAN (lowest first = weakest)
  business_value_score         mean=0.94
  completeness_score           mean=1.77
  testability_score            mean=2.23
  clarity_score                mean=2.55
  scope_risk_score             mean=4.26


In [6]:
print(" ISSUE TAG FREQUENCY\n")

from collections import Counter
all_tags = []
for tag_list in df['issue_tags']:
    all_tags.extend(tag_list)

tag_counter = Counter(all_tags)
total_stories = len(df)

print(f"{'Tag':<32} {'Count':>8} {'Share':>8}")
print("-" * 50)
for tag, count in tag_counter.most_common():
    pct = count / total_stories * 100
    print(f"{tag:<32} {count:>8,} {pct:>7.1f}%")

no_issues = (df['issue_tags_count'] == 0).sum()
print(f"\nStories with NO issue tags at all: {no_issues:,} ({no_issues/total_stories*100:.1f}%)")
many_issues = (df['issue_tags_count'] >= 8).sum()
print(f"Stories with 8 or more issue tags:  {many_issues:,} ({many_issues/total_stories*100:.1f}%)")

 ISSUE TAG FREQUENCY

Tag                                 Count    Share
--------------------------------------------------
missing_acceptance_criteria        31,166    99.3%
missing_reason                     30,110    95.9%
weak_means                         29,471    93.9%
weak_role                          28,383    90.4%
has_implementation_hint            10,647    33.9%
missing_estimate                    9,661    30.8%
non_fibonacci_estimate              7,500    23.9%
has_vague_words                     4,192    13.4%
missing_description                 4,102    13.1%
non_atomic                          2,973     9.5%
high_scope_risk                     1,019     3.2%
duplicate_in_project                  540     1.7%
extreme_scope_risk                     75     0.2%
markup_only                            35     0.1%

Stories with NO issue tags at all: 12 (0.0%)
Stories with 8 or more issue tags:  386 (1.2%)


In [7]:
print(" Average Overall Quality Score BY Project \n")

project_summary = df.groupby('Project_Name').agg(
    story_count=('ID', 'count'),
    avg_overall=('overall_quality_score', 'mean'),
    median_overall=('overall_quality_score', 'median'),
    avg_clarity=('clarity_score', 'mean'),
    avg_completeness=('completeness_score', 'mean'),
    avg_testability=('testability_score', 'mean'),
    avg_business_value=('business_value_score', 'mean'),
    avg_scope_risk=('scope_risk_score', 'mean'),
).reset_index()

project_summary = project_summary[project_summary['story_count'] >= 100].copy()
project_summary = project_summary.sort_values('avg_overall', ascending=False).round(2)
print(project_summary.to_string(index=False))

 Average Overall Quality Score BY Project 

            Project_Name  story_count  avg_overall  median_overall  avg_clarity  avg_completeness  avg_testability  avg_business_value  avg_scope_risk
        MongoDB Compass           175         3.31            3.45         4.07              3.33             3.54                2.75            3.67
   Hyperledger Indy Node          349         2.44            2.13         3.26              2.32             2.57                1.66            3.71
     DotNetNuke Platform          402         2.44            2.29         3.00              2.31             2.49                1.42            4.29
               Spring XD         2593         2.18            2.24         2.91              1.89             2.28                1.29            4.90
          Sonatype Nexus          243         2.12            2.13         2.54              2.23             2.63                1.00            4.00
     Appcelerator Studio          914         2.09

In [8]:
import plotly.express as px
fig = px.histogram(
    df,
    x='overall_quality_score',
    nbins=50,
    title=f'Overall Quality Score Distribution (n = {len(df):,})',
    labels={'overall_quality_score': 'Overall quality score (0 to 5)',
            'count': 'Number of stories'},
)

for boundary, label in [(4.0, 'Very good'), (3.0, 'Good'), (2.0, 'Average'), (1.0, 'Poor')]:
    fig.add_vline(x=boundary, line_dash='dash', line_color='grey',
                  annotation_text=label, annotation_position='top')

fig.update_layout(template='plotly_white', bargap=0.1, height=500)
fig.show()

In [9]:
# Saveed the scored CSV. This is the file Power BI will connect to.

df_save = df.drop(columns=['issue_tags'])
df_save.to_csv(OUT_PATH, index=False, encoding='utf-8')
import os
size_mb = os.path.getsize(OUT_PATH) / 1024**2
print(f"Saved: {OUT_PATH}")
print(f"Rows:  {len(df_save):,}")
print(f"Cols:  {len(df_save.columns)}")
print(f"Size:  {size_mb:.1f} MB on disk")
print(f"\nNew scoring-related columns saved:")
score_cols = [c for c in df_save.columns
              if 'score' in c or 'band' in c or 'issue_tags' in c]
for c in score_cols:
    print(f"  - {c}")

Saved: ..\data\processed\05_user_stories_scored.csv
Rows:  31,394
Cols:  55
Size:  46.5 MB on disk

New scoring-related columns saved:
  - clarity_score
  - completeness_score
  - testability_score
  - business_value_score
  - scope_risk_score
  - overall_quality_score
  - score_band
  - issue_tags_str
  - issue_tags_count


## Summary

**What this notebook did:**
- Imported the scoring module from `src/scoring.py`.
- Applied the 5-dimension scoring framework (Cohn + Lucassen + Squale + Challa) to all 31,394 stories in 7 seconds.
- Added 10 new columns: 5 dimension scores, an overall quality score, a readable score band, an issue tag list, a pipe-joined tag string for CSV storage, and an issue tag count.
- Produced the first quantitative quality signal across the whole dataset.
- Saved `data/processed/05_user_stories_scored.csv` (46.5 MB), which is the Power BI source file.

**Distribution highlights:**

| Metric | Value |
|---|---:|
| Overall score mean | 1.93 |
| Overall score median | 2.03 |
| Stories scoring Very good or Good | 1,467 (4.7%) |
| Stories scoring Poor or Very poor | 15,567 (49.6%) |
| Stories with zero issue tags | 12 (0.04%) |

**Per-dimension ranking, weakest to strongest:**
- Business value: 0.94 (almost no story explains "why")
- Completeness: 1.77
- Testability: 2.23
- Clarity: 2.55
- Scope risk: 4.26 (Story Point coverage is the saving grace)

**Top three issue tags:**
- missing_acceptance_criteria: 99.3%
- missing_reason: 95.9%
- weak_means: 93.9%

**Per-project highlights:**
- Highest: MongoDB Compass (3.31), Hyperledger Indy Node (2.44), DotNetNuke (2.44)
- Lowest: Apache Usergrid (1.58), Alloy Framework (1.65), Apache MXNet (1.75)
- Lsstcorp (62% of the dataset) scores 1.88, pulling the dataset average down.

**What is next:** notebook 06 picks a sample of stories (200 to 500 to control cost) and asks an LLM (OpenAI API) for an independent quality assessment using the same QUS-based criteria. We then compare the rule-based scores from this notebook with the LLM scores to see where they agree and where they diverge.